Week 4
Build Analytics API
Add indexes
Benchmark queries

Week 5
Add Prometheus
Structured logs
Health checks
Load testing using k6


Week 6 (Advanced)
Replace LISTEN/NOTIFY with Kafka + Debezium
Add Redis caching
Add OpenTelemetry tracing

Week 5 is about making your project look like something an SRE, backend engineer, or platform engineer would actually deploy.

At the end of Week 5 you'll have:

```text
                    Prometheus
                         ▲
                         │
                  /metrics
                         │
                         ▼

Analytics Service      Order Service
       ▲                     ▲
       │                     │
       │                     │
 Structured Logs      Structured Logs
       │                     │
       ▼                     ▼

     Health Checks (/health)

                         ▲
                         │

                        k6
                  Load Testing
```

---

# Deliverables

## 1. Prometheus Metrics

Track:

| Metric               | Type      |
| -------------------- | --------- |
| Requests count       | Counter   |
| Request latency      | Histogram |
| Orders created       | Counter   |
| CDC events processed | Counter   |
| Sync failures        | Counter   |
| DB query duration    | Histogram |

---

# Add Dependencies

Order Service:

```bash
go get github.com/prometheus/client_golang/prometheus
go get github.com/prometheus/client_golang/prometheus/promhttp
```

Analytics Service:

```bash
go get github.com/prometheus/client_golang/prometheus
go get github.com/prometheus/client_golang/prometheus/promhttp
```

---

# Metrics Package

## Path

```text
internal/metrics/metrics.go
```

```go
package metrics

import (
	"github.com/prometheus/client_golang/prometheus"
)

var HttpRequests = prometheus.NewCounterVec(
	prometheus.CounterOpts{
		Name: "http_requests_total",
		Help: "Total HTTP Requests",
	},
	[]string{"method", "path"},
)

var HttpDuration = prometheus.NewHistogramVec(
	prometheus.HistogramOpts{
		Name:    "http_request_duration_seconds",
		Help:    "Request latency",
		Buckets: prometheus.DefBuckets,
	},
	[]string{"method", "path"},
)

func Init() {

	prometheus.MustRegister(
		HttpRequests,
		HttpDuration,
	)
}
```

---

# Middleware

## Path

```text
internal/middleware/prometheus.go
```

```go
package middleware

import (
	"net/http"
	"time"

	"yourmodule/internal/metrics"
)

func Metrics(next http.Handler) http.Handler {

	return http.HandlerFunc(
		func(w http.ResponseWriter, r *http.Request) {

			start := time.Now()

			next.ServeHTTP(w, r)

			metrics.HttpRequests.
				WithLabelValues(
					r.Method,
					r.URL.Path,
				).
				Inc()

			metrics.HttpDuration.
				WithLabelValues(
					r.Method,
					r.URL.Path,
				).
				Observe(
					time.Since(start).Seconds(),
				)
		},
	)
}
```

---

# Expose Metrics

In `main.go`

```go
import (
    "github.com/prometheus/client_golang/prometheus/promhttp"
)
```

```go
metrics.Init()

r.Use(middleware.Metrics)

r.Handle(
	"/metrics",
	promhttp.Handler(),
)
```

---

# Prometheus Server

## Path

```text
infra/prometheus/prometheus.yml
```

```yaml
global:
  scrape_interval: 5s

scrape_configs:

  - job_name: order-service

    static_configs:
      - targets:
          - order-service:8080

  - job_name: analytics-service

    static_configs:
      - targets:
          - analytics-service:8081
```

---

# Prometheus Docker

Add to compose:

```yaml
prometheus:
  image: prom/prometheus

  container_name: prometheus

  ports:
    - "9090:9090"

  volumes:
    - ./infra/prometheus/prometheus.yml:/etc/prometheus/prometheus.yml

  restart: unless-stopped
```

---

# 2. Structured Logging

Instead of:

```go
log.Println("received order")
```

use JSON logs.

Install:

```bash
go get go.uber.org/zap
```

---

## Path

```text
internal/logger/logger.go
```

```go
package logger

import "go.uber.org/zap"

var Log *zap.Logger

func Init() error {

	var err error

	Log, err = zap.NewProduction()

	return err
}
```

---

Usage:

```go
logger.Log.Info(
	"order_created",

	zap.Int64(
		"order_id",
		orderID,
	),

	zap.Int(
		"restaurant_id",
		req.RestaurantID,
	),
)
```

Output:

```json
{
  "level":"info",
  "msg":"order_created",
  "order_id":123,
  "restaurant_id":1
}
```

---

# 3. Health Checks

Every service should expose:

```text
GET /health
```

---

## Path

```text
internal/handler/health.go
```

```go
package handler

import (
	"net/http"
)

func Health(
	w http.ResponseWriter,
	r *http.Request,
) {

	w.WriteHeader(http.StatusOK)

	w.Write(
		[]byte("healthy"),
	)
}
```

Register:

```go
r.Get(
	"/health",
	handler.Health,
)
```

---

# Database Health Check

A better version:

```go
func Health(
	pool *pgxpool.Pool,
) http.HandlerFunc {

	return func(
		w http.ResponseWriter,
		r *http.Request,
	) {

		err := pool.Ping(
			r.Context(),
		)

		if err != nil {

			http.Error(
				w,
				"db unavailable",
				http.StatusServiceUnavailable,
			)

			return
		}

		w.WriteHeader(
			http.StatusOK,
		)

		w.Write(
			[]byte("healthy"),
		)
	}
}
```

---

# Docker Healthcheck

Order Service:

```yaml
healthcheck:
  test:
    [
      "CMD",
      "wget",
      "--spider",
      "-q",
      "http://localhost:8080/health"
    ]
  interval: 10s
  timeout: 5s
  retries: 3
```

Analytics Service:

```yaml
healthcheck:
  test:
    [
      "CMD",
      "wget",
      "--spider",
      "-q",
      "http://localhost:8081/health"
    ]
  interval: 10s
  timeout: 5s
  retries: 3
```

---

# 4. k6 Load Testing

Install:

```bash
brew install k6
```

or

```bash
sudo apt install k6
```

---

## Path

```text
tests/load/order_test.js
```

```javascript
import http from "k6/http";

import { check } from "k6";

export const options = {
  vus: 50,
  duration: "60s",
};

export default function () {

  const payload = JSON.stringify({
    customer_id: 1,
    restaurant_id: 1,
    item_id: 1,
    quantity: 2,
    price: 199,
  });

  const params = {
    headers: {
      "Content-Type": "application/json",
    },
  };

  const res = http.post(
    "http://localhost:8080/orders",
    payload,
    params,
  );

  check(res, {
    "status is 201": (r) => r.status === 201,
  });
}
```

Run:

```bash
k6 run tests/load/order_test.js
```

---

# Benchmark Analytics

## Path

```text
tests/load/dashboard_test.js
```

```javascript
import http from "k6/http";

export const options = {
  vus: 100,
  duration: "30s",
};

export default function () {

  http.get(
    "http://localhost:8081/analytics/dashboard?restaurant_id=1",
  );
}
```

Run:

```bash
k6 run tests/load/dashboard_test.js
```

---

# Expected Results

Order API:

```text
P95 < 100ms
P99 < 200ms
```

Analytics API:

```text
P95 < 20ms
```

because all analytics queries hit:

```sql
fact_order_sales
```

with no joins.

---

# Final Project Structure

```text
food-delivery-analytics/

├── services/
│   ├── order-service/
│   ├── cdc-worker/
│   └── analytics-service/
│
├── database/
│   ├── oltp/
│   └── olap/
│
├── infra/
│   └── prometheus/
│
├── tests/
│   └── load/
│
├── docker-compose.yml
│
└── README.md
```

---

## What This Demonstrates on a Resume

This project now showcases:

* Go backend development
* PostgreSQL schema design (3NF + OLAP)
* Database migrations
* Transaction handling
* Context propagation & timeouts
* PostgreSQL LISTEN/NOTIFY CDC
* Background workers
* Data warehousing concepts
* Analytics API design
* Prometheus observability
* Structured logging with Zap
* Docker & Docker Compose
* Load testing with k6
* Query optimization & indexing

That's a strong end-to-end backend/data engineering project for an SDE-1 or early-career backend role.
